# Avaliação do Agente — versão multi-provider

Mesma estrutura do `run_eval.ipynb` original, mas com captura de tokens
**provider-agnóstica** via callback manual. Funciona para OpenAI, Anthropic,
Google e qualquer outro provider langchain que popule `usage_metadata`.

Defaults: **anthropic / claude-haiku-4-5** + edital BNDES.

In [1]:
import os, sys, time
from pathlib import Path
from datetime import datetime

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from agent.agent import build_agent, ask
from langchain_core.callbacks import BaseCallbackHandler

print(f"cwd:  {Path.cwd()}")
print(f"root: {ROOT}")

cwd:  /home/julio/Documentos/tcc_GENAI/v8/edital-assistant/evals
root: /home/julio/Documentos/tcc_GENAI/v8/edital-assistant


## TokenTracker — callback provider-agnóstico

Registra `input_tokens` e `output_tokens` de toda invocação LLM dentro de seu
escopo. Lê primeiro `llm_output.token_usage` (formato antigo, OpenAI-style) e
fallback para `message.usage_metadata` (formato novo, Anthropic/Google/etc).

In [2]:
class TokenTracker(BaseCallbackHandler):
    def __init__(self):
        self.reset()

    def reset(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.n_calls = 0

    def on_llm_end(self, response, **kwargs):
        self.n_calls += 1

        # 1. formato antigo (langchain-openai): llm_output.token_usage
        usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
        if usage:
            self.input_tokens  += usage.get("prompt_tokens", 0) or 0
            self.output_tokens += usage.get("completion_tokens", 0) or 0
            return

        # 2. formato novo (padrão LangChain): message.usage_metadata
        for gen_list in response.generations:
            for gen in gen_list:
                msg = getattr(gen, "message", None)
                um = getattr(msg, "usage_metadata", None) if msg else None
                if um:
                    self.input_tokens  += um.get("input_tokens", 0) or 0
                    self.output_tokens += um.get("output_tokens", 0) or 0
                    break

## Configuração

In [ ]:
EDITAIS = {
    "bndes":     1,
    "cvm":       2,
    "petrobras": 3,
}

EDITAL_NOME = "bndes"
PROVIDER = "deepseek"
MODEL     = "deepseek-chat"

# preço em USD por 1M de tokens (input, output)
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.60),
    "openai/gpt-5.4-mini":            (0.25,  2.00),
    "openai/gpt-5.4":                 (1.25, 10.00),
    "openai/gpt-5.5":                 (5.00, 30.00),
    "deepseek/deepseek-v4-flash":     (0.14,  0.028,   0.28),
    "deepseek/deepseek-v4-pro":       (1.74,  0.145,   3.48),
    "anthropic/claude-haiku-4-5":     (1.00,  5.00),
    "anthropic/claude-sonnet-4-6":    (3.00, 15.00),
    "anthropic/claude-opus-4-7":      (5.00, 25.00),
}

EDITAL_ID = EDITAIS[EDITAL_NOME]
ARQ_PERGUNTAS = Path("perguntas") / f"{EDITAL_NOME}.xlsx"
PRECO_IN, PRECO_OUT = PRECOS[f"{PROVIDER}/{MODEL}"]

print(f"Edital:    {EDITAL_NOME} (id={EDITAL_ID})")
print(f"Modelo:    {PROVIDER}/{MODEL}")
print(f"Preço:     ${PRECO_IN}/M input  ${PRECO_OUT}/M output")
print(f"Perguntas: {ARQ_PERGUNTAS}")

Edital:    bndes (id=1)
Modelo:    deepseek/deepseek-chat
Preço:     $0.28/M input  $0.42/M output
Perguntas: perguntas/bndes.xlsx


## Carregar perguntas

In [4]:
df_q = pd.read_excel(ARQ_PERGUNTAS)
df_q = df_q.dropna(subset=["pergunta"])
df_q = df_q[df_q["pergunta"].str.strip() != ""].reset_index(drop=True)
print(f"{len(df_q)} perguntas carregadas")
df_q.head()

50 perguntas carregadas


,id,categoria,pergunta
0,1,Dados sobre as inscrições,Qual o período de inscrição para o concurso BN...
1,2,Dados sobre as inscrições,Qual o valor da taxa de inscrição do concurso ...
2,3,Dados sobre as inscrições,Em qual site o candidato deve fazer a inscriçã...
3,4,Dados sobre as inscrições,Quais são as formas de pagamento da taxa de in...
4,5,Dados sobre as inscrições,Qual é o prazo-limite de pagamento da inscriçã...


## Construir agente + anexar tracker

`with_config(callbacks=[tracker])` injeta o tracker nas invocações dos dois
LLMs do agente (o principal com tools e o do self-check). Não modifica
`agent.py` — atua via config dinâmica.

In [5]:
agente = build_agent(provider=PROVIDER, model=MODEL)

tracker = TokenTracker()
agente.llm       = agente.llm.with_config(callbacks=[tracker])
agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

print(f"Agente: {PROVIDER}/{MODEL}")
print(f"Tracker anexado em llm e llm_check")

Agente: deepseek/deepseek-chat
Tracker anexado em llm e llm_check


## Rodar as 50 perguntas

`chat_history=[]` em todas as chamadas → memória off. Tracker é zerado antes
de cada pergunta para isolar o consumo do turno.

In [6]:
resultados = []

for i, row in df_q.iterrows():
    pergunta = row["pergunta"]
    t0 = time.time()
    erro = None
    resposta = None

    tracker.reset()
    try:
        resposta = ask(
            agent=agente,
            question=pergunta,
            chat_history=[],
            edital_id_ativo=EDITAL_ID,
        )
    except Exception as e:
        erro = str(e)

    in_tok    = tracker.input_tokens
    out_tok   = tracker.output_tokens
    n_calls   = tracker.n_calls
    custo_usd = in_tok / 1_000_000 * PRECO_IN + out_tok / 1_000_000 * PRECO_OUT
    latencia  = round(time.time() - t0, 2)

    resultados.append({
        "id":            row["id"],
        "categoria":     row.get("categoria", ""),
        "pergunta":      pergunta,
        "resposta":      resposta,
        "input_tokens":  in_tok,
        "output_tokens": out_tok,
        "total_tokens":  in_tok + out_tok,
        "n_invocacoes":  n_calls,
        "custo_usd":     round(custo_usd, 6),
        "latencia_s":    latencia,
        "erro":          erro,
    })

    flag = "ERRO" if erro else "ok"
    print(f"[{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
          f"calls={n_calls}  in={in_tok:>5}  out={out_tok:>4}")

df_r = pd.DataFrame(resultados)
print(f"\n{len(df_r)} respostas coletadas")

Carregando BGE-M3 (primeira vez pode demorar)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 41638.47it/s]


[ 1/50]   ok   10.6s  calls=2  in= 8804  out= 219
[ 2/50]   ok   5.25s  calls=2  in= 8441  out= 290
[ 3/50]   ok   6.07s  calls=3  in=21343  out= 260
[ 4/50]   ok   5.73s  calls=2  in= 8221  out= 302
[ 5/50]   ok   5.54s  calls=2  in= 9044  out= 298
[ 6/50]   ok   6.96s  calls=2  in= 9030  out= 424
[ 7/50]   ok   6.35s  calls=2  in= 8991  out= 357
[ 8/50]   ok   6.34s  calls=2  in= 8209  out= 364
[ 9/50]   ok   5.94s  calls=2  in= 8369  out= 353
[10/50]   ok   6.46s  calls=2  in= 8587  out= 366
[11/50]   ok   5.52s  calls=2  in= 8131  out= 297
[12/50]   ok   6.45s  calls=2  in= 8917  out= 378
[13/50]   ok   4.91s  calls=2  in= 9274  out= 240
[14/50]   ok   7.42s  calls=2  in= 8531  out= 438
[15/50]   ok   5.69s  calls=2  in= 9569  out= 280
[16/50]   ok   6.45s  calls=2  in= 9029  out= 377
[17/50]   ok  16.38s  calls=5  in=44822  out= 915
[18/50]   ok   7.35s  calls=2  in= 8859  out= 456
[19/50]   ok   9.72s  calls=4  in=23715  out= 474
[20/50]   ok   8.63s  calls=4  in=24216  out= 423


## Salvar

In [7]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
slug_modelo = MODEL.replace("/", "_").replace(":", "_")
nome_base = f"{EDITAL_NOME}__{PROVIDER}__{slug_modelo}__{ts}"

Path("resultados").mkdir(exist_ok=True)
arq = Path("resultados") / f"{nome_base}.xlsx"
df_r.to_excel(arq, index=False)
print(f"Salvo: {arq}")

Salvo: resultados/bndes__deepseek__deepseek-chat__20260429_214233.xlsx


## Sumário

In [8]:
total_in   = int(df_r["input_tokens"].sum())
total_out  = int(df_r["output_tokens"].sum())
total_cost = float(df_r["custo_usd"].sum())
n_erros    = int(df_r["erro"].notna().sum())
lat_media  = float(df_r["latencia_s"].mean())
lat_p95    = float(df_r["latencia_s"].quantile(0.95))
calls_med  = float(df_r["n_invocacoes"].mean())

print(f"Perguntas:        {len(df_r)}  ({n_erros} com erro)")
print(f"Tokens entrada:   {total_in:,}")
print(f"Tokens saída:     {total_out:,}")
print(f"Tokens totais:    {total_in + total_out:,}")
print(f"Custo medido:     US$ {total_cost:.4f}   ({PROVIDER}/{MODEL})")
print(f"Latência média:   {lat_media:.1f}s")
print(f"Latência p95:     {lat_p95:.1f}s")
print(f"Invocações médias por pergunta: {calls_med:.1f}")

Perguntas:        50  (0 com erro)
Tokens entrada:   685,612
Tokens saída:     21,381
Tokens totais:    706,993
Custo medido:     US$ 0.2009   (deepseek/deepseek-chat)
Latência média:   7.8s
Latência p95:     15.8s
Invocações médias por pergunta: 2.4


## Extrapolação para os outros modelos

A partir dos tokens medidos **agora** (com o tokenizer deste modelo),
estima o custo dos outros. Tokenizers diferem entre providers — números
reais podem variar ~10–35%.

In [9]:
linhas = []
for nome, (pi, po) in PRECOS.items():
    custo_1ed = total_in / 1_000_000 * pi + total_out / 1_000_000 * po
    linhas.append({
        "modelo":              nome,
        "preco_in_usd_p_1M":   pi,
        "preco_out_usd_p_1M":  po,
        "custo_50q_1edital":   round(custo_1ed, 4),
        "custo_50q_3editais":  round(custo_1ed * 3, 4),
    })

df_custos = (
    pd.DataFrame(linhas)
    .sort_values("custo_50q_1edital")
    .reset_index(drop=True)
)
df_custos

,modelo,preco_in_usd_p_1M,preco_out_usd_p_1M,custo_50q_1edital,custo_50q_3editais
0,openai/gpt-4o-mini,0.150,0.60,0.1157,0.3470
1,deepseek/deepseek-chat,0.280,0.42,0.2010,0.6029
2,openai/gpt-5.4-mini,0.250,2.00,0.2142,0.6425
3,deepseek/deepseek-v4-flash,0.300,0.50,0.2164,0.6491
4,deepseek/deepseek-v4-pro,0.435,0.87,0.3168,0.9505
5,anthropic/claude-haiku-4-5,1.000,5.00,0.7925,2.3776
6,openai/gpt-5.4,1.250,10.00,1.0708,3.2125
7,anthropic/claude-sonnet-4-6,3.000,15.00,2.3776,7.1327
8,anthropic/claude-opus-4-7,5.000,25.00,3.9626,11.8878
9,openai/gpt-5.5,5.000,30.00,4.0695,12.2085


## Inspecionar respostas

In [10]:
for _, row in df_r.head(5).iterrows():
    print(f"--- Q{int(row['id'])} ---")
    print(f"P: {row['pergunta']}")
    print(f"R: {row['resposta']}")
    print(f"   tokens={row['total_tokens']}  calls={row['n_invocacoes']}  latencia={row['latencia_s']}s\n")

--- Q1 ---
P: Qual o período de inscrição para o concurso BNDES?
R: **Período de inscrição do BNDES (01/2024):**

O período de inscrição foi de **26 de julho a 19 de agosto de 2024**, com inscrições realizadas entre **10h do dia 26/07 e 23h59 do dia 19/08/2024** na página da Fundação Cesgranrio (www.cesgranrio.org.br).

**Status atual:** Hoje é **29/04/2026**. O período de inscrição **já encerrou** há mais de um ano e meio. Este concurso não está mais aberto para novas inscrições.

ref: edital [6.2.1, 6.3.1, ANEXO III]
   tokens=11259  calls=2  latencia=9.55s

--- Q2 ---
P: Qual o valor da taxa de inscrição do concurso BNDES?
R: A **taxa de inscrição do concurso BNDES é de R$ 110,00 (cento e dez reais)**.

O pagamento pode ser feito de duas formas:
1. **Boleto Bancário** — em qualquer banco até a data de vencimento
2. **Cobrança PIX** — usando a opção "Copia e Cola" ou QR Code gerado

**Pontos importantes:**
- A inscrição só é válida após a confirmação do pagamento até a data de vencim